# 3.3 — F1 Score

F1 score combines precision and recall into a single number using the harmonic mean.

**Formula:** `F1 = 2 × (Precision × Recall) / (Precision + Recall)`

> **Why harmonic mean?** Simple average can be fooled — if precision=1.0 and recall=0.01, simple average gives 0.505 (looks okay). F1 gives 0.02 (correctly reveals it's terrible). Both precision AND recall must be good for F1 to be good.

**Range:** 0 to 1. Higher is better.

**Use when:** classes are imbalanced and you need both precision and recall to be good.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    classification_report, accuracy_score
)

# Load and prepare Titanic
df = sns.load_dataset('titanic')
cols = ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
df = df[cols].copy()
df['age'] = df['age'].fillna(df['age'].median())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])
df['family_size'] = df['sibsp'] + df['parch'] + 1
df['is_alone'] = (df['family_size'] == 1).astype(int)
df['sex'] = LabelEncoder().fit_transform(df['sex'])
df = pd.get_dummies(df, columns=['embarked'], drop_first=True)

X = df.drop(columns=['survived'])
y = df['survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

model = LogisticRegression(random_state=42)
model.fit(X_train_s, y_train)
y_pred = model.predict(X_test_s)

print("Setup complete.")

## Why Simple Average Fails

In [ ]:
# Demonstrate why harmonic mean is better than simple average
examples = [
    (0.90, 0.90, "balanced"),
    (1.00, 0.01, "high precision, terrible recall"),
    (0.01, 1.00, "terrible precision, high recall"),
    (0.70, 0.80, "both decent"),
]

print(f"{'Precision':>10}  {'Recall':>8}  {'Simple avg':>11}  {'F1':>6}  Case")
print("-" * 65)
for p, r, label in examples:
    simple = (p + r) / 2
    f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0
    print(f"{p:>10.2f}  {r:>8.2f}  {simple:>11.3f}  {f1:>6.3f}  {label}")

## F1 Score — Step by Step From Scratch

In [ ]:
# Step 1 — get precision and recall
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)

# Step 2 — calculate F1 manually
f1_manual = 2 * (precision * recall) / (precision + recall)

# Step 3 — verify with sklearn
f1_sklearn = f1_score(y_test, y_pred)

print(f"Precision:     {precision:.3f}")
print(f"Recall:        {recall:.3f}")
print(f"F1 (manual):   {f1_manual:.3f}")
print(f"F1 (sklearn):  {f1_sklearn:.3f}")
print(f"\nF1 is closer to the lower value ({min(precision,recall):.3f}) — harmonic mean punishes imbalance")

## Full Classification Report

Shows precision, recall, F1, and support for every class — run this before reporting any model's performance.

In [ ]:
print(classification_report(y_test, y_pred,
      target_names=['Not survived', 'Survived']))

## Macro vs Weighted Average

- **Macro avg** — F1 per class, then simple average. Treats all classes equally.
- **Weighted avg** — F1 per class, averaged by support. Bigger classes count more.

In [ ]:
f1_macro    = f1_score(y_test, y_pred, average='macro')
f1_weighted = f1_score(y_test, y_pred, average='weighted')
f1_binary   = f1_score(y_test, y_pred)

print(f"Binary F1 (positive class only): {f1_binary:.3f}")
print(f"Macro F1 (equal class weight):   {f1_macro:.3f}")
print(f"Weighted F1 (by support):        {f1_weighted:.3f}")
print()
print("Class support:")
print(y_test.value_counts())
print("\nWeighted avg gives more weight to 'Not survived' (larger class)")

## Accuracy vs F1 — Which is More Honest?

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
f1       = f1_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.3f}")
print(f"F1 Score: {f1:.3f}")
print()
print("Class distribution:")
print((y_test.value_counts(normalize=True) * 100).round(1))
print()
print("Titanic is mildly imbalanced (62/38).")
print("F1 is more honest — it accounts for the imbalance.")
print("For heavily imbalanced data, the gap between accuracy and F1 would be much larger.")

## Decision Rules

| Situation | Metric |
|-----------|--------|
| Balanced classes, quick check | Accuracy |
| Imbalanced classes | F1 score |
| Missing positives very costly | Recall |
| False alarms very costly | Precision |
| Need one number to compare models | F1 or AUC |
| Full report for stakeholders | classification_report |

> **Key insight:** F1 is your default metric for most real-world classification problems. Fraud, disease, defects — all imbalanced. Accuracy will lie. F1 won't.

## Practice Task

In [ ]:
# Q1 — Calculate F1 score using sklearn
# YOUR CODE HERE

# Q2 — Calculate F1 manually using: F1 = 2*(P*R)/(P+R)
# YOUR CODE HERE

# Q3 — Print the full classification report
# YOUR CODE HERE

# Q4 — Why are macro avg and weighted avg F1 slightly different?
# ANSWER:

# Q5 — Accuracy = 0.81, F1 = 0.77 for Titanic.
#       Which is more honest for this dataset and why?
# ANSWER: